In [1]:
# Getting imports and the model setup
import os
import sys
sys.path.insert(0, "../examples/open_deep_research")
from smolagents import OpenAIModel

from dotenv import load_dotenv
load_dotenv()

# model_name = "gpt-4o"
# model_name = "gpt-5.4-mini"
model_name = "Qwen/Qwen3.7-Plus"
# model_name = "Qwen/Qwen3.5-9B"

enable_thinking = False
if model_name in ["gpt-4o", "gpt-5.4-mini"]:
    model = OpenAIModel(
        model_id=model_name,
        api_key=os.environ["OPENAI_API_KEY"],
    )
elif model_name in ["Qwen/Qwen3.7-Plus", "Qwen/Qwen3.5-9B"]:
    enable_thinking = False  # hardcoded (no interactive input) so this notebook can run headlessly via nbconvert
    # Together requires enable_thinking nested inside chat_template_kwargs for vLLM-served open-weight models.
    # client_kwargs timeout bounds a single API call so a stalled/hanging stream fails within 5 minutes
    # instead of hanging indefinitely -- evaluate_agent already catches and records such errors.
    model = OpenAIModel(
        model_id=model_name,
        api_base="https://api.together.ai/v1/",
        api_key=os.environ["TOGETHER_API_KEY"],
        extra_body={"chat_template_kwargs": {"enable_thinking": enable_thinking}},
        client_kwargs={"timeout": 300.0},
    )
    print(f"Using model: {model_name} with thinking enabled: {enable_thinking}")

Using model: Qwen/Qwen3.7-Plus with thinking enabled: False


In [2]:
# Debug check: fail loudly the moment any Together-served step returns reasoning despite
# enable_thinking=False, rather than only noticing it later by eyeballing console output.
from common_setup import assert_no_reasoning

/Users/poorvag/Work/smolagents/.venv/lib/python3.14/site-packages/pydub/utils.py:170: RuntimeWarning: Couldn't find ffmpeg or avconv - defaulting to ffmpeg, but may not work
  warn("Couldn't find ffmpeg or avconv - defaulting to ffmpeg, but may not work", RuntimeWarning)


In [3]:
# Getting the tools setup and the agent setup
from smolagents import CodeAgent
from smolagents.monitoring import LogLevel
from common_setup import build_tools

tools, ti_tool, visualizer = build_tools(model)

agent = CodeAgent(
    tools=tools,
    model=model,
    max_steps=50,
    verbosity_level=LogLevel.DEBUG,
    additional_authorized_imports=["pandas", "numpy", "PIL", "json", "io", "zipfile", "csv", "openpyxl"],
    stream_outputs="Qwen" in model_name,
    step_callbacks=[assert_no_reasoning] if enable_thinking is False else None,
)

## Fermi (RealFP) dataset

RealFP is the ~[allenai/fermi](https://github.com/allenai/fermi) collection of Fermi estimation
problems (Kalyan et al., 2021) -- questions whose answer is a number that can only be
reasonably estimated. Answers are scored with `fermi_scorer`, a continuous 0-1 order-of-magnitude
accuracy metric (1.0 = exact match, decaying to 0 three orders of magnitude off), not GAIA's
exact-match boolean scorer -- see `fermi_setup.py`.

In [4]:
# Load the Fermi RealFP validation split
import pandas as pd
from fermi_setup import load_fermi_dataset

eval_ds = load_fermi_dataset(split="val")
print(f"Loaded {len(eval_ds)} examples")

Loaded 125 examples


In [5]:
# Scoring + eval loop reused from common_setup.py (shared with GAIA), with fermi_scorer and a
# Fermi-specific final-answer formatting instruction plugged in
from common_setup import evaluate_agent
from fermi_setup import fermi_scorer, FERMI_ANSWER_INSTRUCTION

**Full run:** all 125 validation questions for the model selected above. evaluate_agent caches
per-question results under `naive_fermi_{model_name}/`, keyed by index, so reruns only make new
API calls for questions not already cached for that model.

In [6]:
safe_model_name = model_name.replace("/", "_")
results = evaluate_agent(
    agent,
    eval_ds,
    ti_tool,
    visualizer,
    n_samples=None,
    output_file=f"naive_fermi_{safe_model_name}.jsonl",
    pickle_dir=f"naive_fermi_{safe_model_name}",
    scorer=fermi_scorer,
    question_suffix=FERMI_ANSWER_INSTRUCTION,
)


[1/125] (cached) If all but 1 million people on Earth died, how far (on average) would you have to walk to meet someo...
  ✓ | cached

[2/125] (cached) How much water would need to be evaporated to turn Arizona tropical?...
  ✗ | cached

[3/125] (cached) If a gallon of paint is used to coat 400 ft2 of walls, how thick, cm, is the paint film?...
  ✓ | cached

[4/125] (cached) How much space would be required to fit 1 individual of each species of plants and animals?...
  ✓ | cached

[5/125] (cached) How many canaries in the world are currently flying?...
  ✓ | cached

[6/125] (cached) How old are you if you are a million hours old?...
  ✓ | cached

[7/125] How many mice could pull the Maus tank?...


╭──────────────────────────────────────────────────── New run ────────────────────────────────────────────────────╮
│                                                                                                                 │
│ How many mice could pull the Maus tank?                                                                         │
│                                                                                                                 │
│ Give your final answer as a number with units matching the question (e.g. "22.3 km", "79 g"; use a bare number  │
│ like "33000" only if the question has no natural unit). Do not include filler words like "about" or "roughly",  │
│ or thousands separators. Write the full number of digits (e.g. "256000000000") rather than word multipliers     │
│ like "million" or "billion".                                                                                    │
│                                                                                                                 │
╰─ OpenAIModel - Qwen/Qwen3.7-Plus ───────────────────────────────────────────────────────────────────────────────╯

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 1 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

Output()

─ Executing parsed code: ──────────────────────────────────────────────────────────────────────────────────────── 
  maus_info = web_search(query="Maus tank weight Panzer VIII")                                                     
  print(maus_info)                                                                                                 
 ─────────────────────────────────────────────────────────────────────────────────────────────────────────────────

...

Output()

─ Executing parsed code: ──────────────────────────────────────────────────────────────────────────────────────── 
  final_answer("1825781")                                                                                          
 ─────────────────────────────────────────────────────────────────────────────────────────────────────────────────

Final answer: 1825781

[Step 25: Duration 3.53 seconds| Input tokens: 441,273 | Output tokens: 1,683]

  ✓ | 118.7s | 25 steps | tokens: TokenUsage(input_tokens=441273, output_tokens=1683, total_tokens=442956)

[76/125] (cached) How many leaves are there in the world?...
  ✓ | cached

[77/125] (cached) How many gallons of water move down the Mississippi River in one day?...
  ✓ | cached

[78/125] (cached) How many multi-link cubes would you need to fill a fridge?...
  ✓ | cached

[79/125] (cached) How many RBCs equal one cc in volume?...
  ✓ | cached

[80/125] (cached) If Donald Trump weighed as much as an average Clydesdale horse, but kept his current density and pro...
  ✓ | cached

[81/125] (cached) What would it cost to build a sailing frigate?...
  ✓ | cached

[82/125] (cached) To prevent the loss of water due to evaporation from a lake, oil is spread to form a thin film. How ...
  ✓ | cached

[83/125] (cached) How many square feet of toilet paper are in the school at any point of time?...
  ✓ | cached

[84/125] (cached) If human skin could photosynthesise with the same efficiency 

In [7]:
df = pd.DataFrame(results)
total = len(df)
avg_score = df["is_correct"].mean()  # continuous 0-1 order-of-magnitude accuracy metric, not a boolean fraction

print("=== Fermi (RealFP) Evaluation Results ===")
print(f"Questions:              {total}")
print(f"Avg score:              {avg_score:.3f}")
print(f"Avg time per question:  {df['time_taken_seconds'].mean():.1f}s")
print(f"Avg steps per question: {df['num_steps'].mean():.1f}")

total_tokens = df["token_counts"].apply(lambda x: x.get("total_tokens", 0)).mean()
print(f"Avg tokens used:        {total_tokens:,.0f}")

print("\nTool usage (total calls across all questions):")
tool_usage_df = pd.DataFrame(df["tool_usage"].tolist()).sum().sort_values(ascending=False)
for tool, count in tool_usage_df.items():
    if count > 0:
        print(f"  {tool}: {int(count)}")

=== Fermi (RealFP) Evaluation Results ===
Questions:              125
Avg score:              0.613
Avg time per question:  117.0s
Avg steps per question: 16.1
Avg tokens used:        299,542

Tool usage (total calls across all questions):
  web_search: 1213
  visit_page: 170
  page_down: 145
  find_on_page_ctrl_f: 137
  final_answer: 120
  find_next: 7
  inspect_file_as_text: 7
  find_archived_url: 3
  page_up: 2
  visualizer: 1
